## Prepare deliberation

In [ ]:
import boto3
import json
from botocore.config import Config
from datetime import datetime

In [ ]:
config = Config(read_timeout=200)
session = boto3.Session(profile_name="default")
client = boto3.client("bedrock-agent-runtime", region_name="us-west-2", config=config)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"deliberation_{timestamp}.json"
FLOW_IDENTIFIER = "UUSQBV36Y1"

deliberation = {}

In [ ]:
def invoke_flow(flowAliasIdentifier, context):
    try:
        response = client.invoke_flow(
            flowIdentifier=FLOW_IDENTIFIER,
            flowAliasIdentifier=flowAliasIdentifier, 
            inputs=[
                {
                    "content": {"document": context},
                    "nodeName": "FlowInputNode",
                    "nodeOutputName": "document"
                }
            ]
        )

        event = response.get('responseStream')

        if event['flowOutputEvent']['nodeName'] == 'FlowOutputNode':
            return event['flowOutputEvent']['content']['document']
        else:
            return False
        
    except Exception as e:
        print(f"An error occured when invoking flow with Alias ID {flowAliasIdentifier}: {e}")
        return False

In [ ]:
def add_context(context, addition):
    return "\n\n".join([context, addition])

In [ ]:
def check_unanimous_agreement(label_lists):
    if not label_lists:
        return []

    length = len(label_lists[0])
    assert all(len(lst) == length for lst in label_lists), "Error: not all label lists are the same length."

    return [
        i for i in range(length)
        if all(label_lists[0][i] == lst[i] for lst in label_lists)
    ]

In [ ]:
from collections import Counter
import math

def check_majority_agreement(label_lists):
    if not label_lists:
        return []

    length = len(label_lists[0])
    assert all(len(lst) == length for lst in label_lists), "Error: not all label lists are the same length."

    majority_indices = []

    num_lists = len(label_lists)
    for i in range(length):
        # Get all labels at position i
        labels_at_i = [lst[i] for lst in label_lists]
        label_counts = Counter(labels_at_i)
        most_common_label, count = label_counts.most_common(1)[0]

        if count >= math.ceil(num_lists / 2):
            majority_indices.append(i)

    return majority_indices

## Start deliberation

In [ ]:
num_unanimous_rounds = 2
num_majority_rounds = 2
num_deliberators = 3

intervention = "Hybrid argumentation is a new paradigm that combines the strengths of traditional deductive and inductive reasoning."
questions = "CQ1, CQ2, CQ3"
context = f"Arguments:\n{intervention}\n\nCritical questions:\n{questions}"

deliberation["intervention"] = intervention
deliberation["questions"] = questions 

### Unanimous rounds

In [ ]:
for _ in range(num_unanimous_rounds):
    for _ in range(num_deliberators):
        response = invoke_flow('', context)
        context = add_context(context, response)
    
    label_lists = []
    for _ in range(num_deliberators):
        labels = invoke_flow(context)
        label_lists.append(labels)

    agreement = check_unanimous_agreement(labels)
    if len(agreement) >= 3:
        deliberation["agreement"] = agreement
        exit()


### Majority rounds

In [ ]:
for round in range(num_majority_rounds):
    for _ in range(num_deliberators):
        response = invoke_flow('', context)
        context = add_context(context, response)
    
    label_lists = []
    for _ in range(num_deliberators):
        labels = invoke_flow(context)
        label_lists.append(labels)

    agreement = check_majority_agreement(labels)
    if len(agreement) >= 3:
        deliberation["agreement"] = agreement
        exit()

### Final round

In [ ]:
import random

for _ in range(num_deliberators):
    response = invoke_flow('', context)
    context = add_context(context, response)
    
label_lists = []
for _ in range(num_deliberators):
    labels = invoke_flow(context)
    label_lists.append(labels)

agreement = check_majority_agreement(labels)

if len(agreement) < 3:
    total_indices = label_lists[0].keys()
    remaining_indices = list(set(total_indices) - set(agreement))
    needed = 3 - len(agreement)

    additional = random.sample(remaining_indices, needed)
    agreement.extend(additional)

deliberation["agreement"] = agreement

### Save deliberation to JSON

In [ ]:
with open(output_file, "w") as f:
    json.dump(deliberation, f, indent=4)

print(f"Saved deliberation to: {output_file}")